In [3]:
import os
import json
import pandas as pd
from PIL import Image

# =========================
# PATHS
# =========================

ROOT = "/kaggle/input/datasets/suhaibalajami/shaheddn-desret-dataset/real_data"

CSV_PATH = os.path.join(ROOT, "annotations.csv")

IMAGES_DIR = os.path.join(ROOT, "images")

# Save labels in writable directory
LABELS_DIR = "/kaggle/working/labels"

# =========================
# LOAD CSV
# =========================

df = pd.read_csv(CSV_PATH)

# =========================
# CLASS MAPPING
# =========================

classes = {
    "human": 0
}

# =========================
# CREATE LABEL DIRECTORIES
# =========================

for split in ["train", "val", "test"]:
    os.makedirs(
        os.path.join(LABELS_DIR, split),
        exist_ok=True
    )

# =========================
# PROCESS EACH ROW
# =========================

for _, row in df.iterrows():

    image_name = row["image"]
    split = row["split"]
    class_name = row["category"]

    # YOLO label file path
    txt_name = image_name.rsplit(".", 1)[0] + ".txt"

    txt_path = os.path.join(
        LABELS_DIR,
        split,
        txt_name
    )

    # =========================
    # HANDLE NEGATIVE SAMPLES
    # =========================

    if class_name == "without human":

        # Create empty txt file
        open(txt_path, "w").close()

        continue

    # =========================
    # CLASS ID
    # =========================

    class_id = classes[class_name]

    # =========================
    # PARSE JSON LABEL
    # =========================

    label = json.loads(row["label"])

    # Your labels are stored as a list
    bbox = label[0]

    x = bbox["x"]
    y = bbox["y"]
    w = bbox["width"]
    h = bbox["height"]

    # =========================
    # OPEN IMAGE
    # =========================

    img_path = os.path.join(
        IMAGES_DIR,
        split,
        image_name
    )

    img = Image.open(img_path)

    img_w, img_h = img.size

    # =========================
    # CONVERT TO YOLO FORMAT
    # =========================

    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h

    width = w / img_w
    height = h / img_h

    # =========================
    # WRITE LABEL FILE
    # =========================

    with open(txt_path, "w") as f:

        f.write(
            f"{class_id} "
            f"{x_center} "
            f"{y_center} "
            f"{width} "
            f"{height}\n"
        )

print("YOLO labels created successfully.")

YOLO labels created successfully.


In [6]:
import os

images = set()
labels = set()

for split in ["train", "val", "test"]:

    img_dir = f"/kaggle/input/datasets/suhaibalajami/shaheddn-desret-dataset/real_data/images/{split}"
    lbl_dir = f"/kaggle/working/labels/{split}"

    for f in os.listdir(img_dir):
        images.add(os.path.splitext(f)[0])

    for f in os.listdir(lbl_dir):
        labels.add(os.path.splitext(f)[0])

print("Missing labels:", images - labels)
print("Extra labels:", labels - images)

Missing labels: set()
Extra labels: set()


In [7]:
import shutil

shutil.make_archive(
    "/kaggle/working/labels",
    'zip',
    "/kaggle/working/labels"
)

print("ZIP created successfully.")

ZIP created successfully.
